In [ ]:
import os, glob, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV

from lightgbm import LGBMRegressor, early_stopping
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)




# column roles (decoded from metadata.csv)
ID       = "TransactionID"          # identifier, not a feature
TARGET   = "TargetValue"            # sale price (USD)
DATE     = "TransactionDate"        # sale date
YEAR_COL = "ManufactureYear"        # year built
HOURS    = "OperationalHoursMeter"  # usage hours
GROUP    = "InventoryGroupDescription"  # broad equipment group (for EDA)
DROP_IDS = ["AssetID"]              # individual-machine id -> drop

# high-cardinality categoricals worth a "how common is it?" (frequency) feature
FREQ_COLS = ["ProductConfigID", "Spec_FullDescriptor", "Spec_BaseClass",
             "FunctionalClassification", "Spec_SubClass"]

# 1. Loading the data:

In [ ]:
# getting the directory/folder path
competition_dir = "/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge"


print("Using folder:", competition_dir)

#getting the files path from the folder
train_path = os.path.join(competition_dir, "train.csv")
test_path  = os.path.join(competition_dir, "test.csv")
print("TRAIN ->", train_path)
print("TEST  ->", test_path)

# finally laoding the files i.e. train.csv and test.csv
train = pd.read_csv(train_path, low_memory=False)
test  = pd.read_csv(test_path,  low_memory=False)

#checking the train and test files shape
print("train:", train.shape, "| test:", test.shape)
train.head()

# 2. Exploratory data analysis:

## 2.1 Overall structure of the dataset:

To know how big it is, what types  of columns are there, and are there any duplicates or not.

In [ ]:
# to know columns data types 
print("\nColumn dtypes:")
print(train.dtypes.value_counts())

#to know duplicate records
print("\nExact duplicate rows :", train.duplicated().sum())
print("Duplicate TransactionID:", train[ID].duplicated().sum())

**Inference: it seen that there are no duplicate records and there are only three types of data types i.e. integer, float and object**

In [ ]:
# to know per column data type and non-null count in one shot
train.info(show_counts=True)

***Inference : Mostly integer and float datatypes are present in first 7 columns and rest are object types, and the columns whose non null entry counts are less 138700 have missing values***

## 2.2 Numerical and categorical columns summary:

In [ ]:
print("Numeric columns summary:")
display(train.describe().T)
print("\nCategorical columns summary (top value + frequency):")
display(train.describe(include='object').T)

## 2.3 Column profiling:

*A single table describing every column: its type, how many unique values, how much is missing, an example value, and the role we assign it (identifier / target / date / numeric / ordinal / categorical / high-cardinality / near-empty).*

In [ ]:
ORDINAL_COLS   = ["UtilizationTier", "AssetScaleFactor"]   # have a natural order
HIGHCARD_LIMIT = 50   # >50 distinct string levels = 'high cardinality'

def assign_role(col, s):
    miss = s.isna().mean()
    nun  = s.nunique(dropna=True)
    if col == ID:                       return "identifier"
    if col == TARGET:                   return "target"
    if col == DATE:                     return "datetime"
    if nun <= 1:                        return "constant (drop)"
    if miss > 0.99:                     return "near-empty (drop)"
    if col == "AssetID":                return "identifier (high-card)"
    if col in [YEAR_COL, HOURS]:        return "numeric"
    if col in ORDINAL_COLS:             return "ordinal categorical"
    if pd.api.types.is_numeric_dtype(s) and nun > HIGHCARD_LIMIT:
        return "high-cardinality (coded)"
    if nun > HIGHCARD_LIMIT:            return "high-cardinality categorical"
    return "categorical (nominal)"

rows = []
for c in train.columns:
    s = train[c]
    rows.append({
        "column": c,
        "dtype": str(s.dtype),
        "n_unique": s.nunique(dropna=True),
        "pct_missing": round(100*s.isna().mean(), 1),
        "example": s.dropna().iloc[0] if s.notna().any() else np.nan,
        "role": assign_role(c, s),
    })
profile = pd.DataFrame(rows).sort_values(["role", "n_unique"], ascending=[True, False])
display(profile.reset_index(drop=True))
print("\nRole counts:")
print(profile["role"].value_counts())

identifier / near-empty / constant → drop (no predictive value).
numeric → impute + maybe scale.
ordinal categorical → encode respecting order (Low<Medium<High, Mini<...<Large).
high-cardinality → ordinal-encode for trees, add frequency features; avoid one-hot (would explode to thousands of columns).
nominal categorical → standard encoding

## 2.4 Row profiling

In [ ]:
row_missing = train.isna().mean(axis=1)
plt.figure(figsize=(9,4))
sns.histplot(row_missing, bins=40, color="#4C72B0")
plt.xlabel("fraction of columns missing in a row"); plt.title("Missingness per row")
plt.tight_layout(); plt.show()
print("Rows >50% empty:", int((row_missing > 0.5).sum()),
      "({:.1f}%)".format(100*(row_missing > 0.5).mean()))

*Inference: Many rows have many empty values in the specification columns, but that is not an error in the data. For some machines, some features do not apply at all. For example, a machine may not have a blade width specification. So that column will be blank for that row.*

## 2.5 Knowing the target variable i.e. TargetValue column

We look at its shape with four complementary plots and decide whether to transform it.

In [ ]:
t = train[TARGET]
print("min={:.0f}  median={:.0f}  mean={:.0f}  max={:.0f}".format(t.min(), t.median(), t.mean(), t.max()))
print("skew(raw)  = {:.2f}   (0 = symmetric, >1 = strong right tail)".format(skew(t)))
print("skew(log)  = {:.2f}".format(skew(np.log1p(t))))
print("kurtosis   = {:.2f}".format(kurtosis(t)))

fig, ax = plt.subplots(2, 2, figsize=(13, 8))
sns.histplot(t, bins=60, kde=True, ax=ax[0,0], color="#C44E52"); ax[0,0].set_title("Histogram + KDE (raw price)")
sns.histplot(np.log1p(t), bins=60, kde=True, ax=ax[0,1], color="#55A868"); ax[0,1].set_title("Histogram + KDE (log price)")
sns.boxplot(x=t, ax=ax[1,0], color="#C44E52"); ax[1,0].set_title("Boxplot (raw) — long upper tail = outliers")
sns.violinplot(x=np.log1p(t), ax=ax[1,1], color="#55A868"); ax[1,1].set_title("Violinplot (log) — near-symmetric")
plt.tight_layout(); plt.show()

*Inference: Raw price is right-skewed (skew ≈ +1) with a long upper tail of expensive machines; the log makes it almost symmetric (skew ≈ 0). Since the metric is RMSLE (error in log space), we will model log(1+TargetValue) — this both matches the metric and stabilises variance.*

## 2.6 Univariate analysis — key numeric features

### ManufactureYear

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,4))
sns.histplot(train[YEAR_COL], bins=80, ax=ax[0], color="#8172B3")
ax[0].set_title("ManufactureYear (raw) — spike at ~1001 is a placeholder")
valid_year = train[YEAR_COL].where(train[YEAR_COL] > 1900)
sns.histplot(valid_year, bins=60, ax=ax[1], color="#8172B3")
ax[1].set_title("ManufactureYear (valid years only)")
plt.tight_layout(); plt.show()
print("Rows with ManufactureYear == 1001 (unknown):", int((train[YEAR_COL] == 1001).sum()))

OperationalHoursMeter

In [ ]:
h = train[HOURS]
print("missing = {} ({:.1f}%) | exactly 0 = {} | >0 median = {:.0f}".format(
      h.isna().sum(), 100*h.isna().mean(), int((h==0).sum()), h[h>0].median()))
fig, ax = plt.subplots(1, 2, figsize=(13,4))
sns.histplot(h[h>0], bins=60, ax=ax[0], color="#4C72B0"); ax[0].set_title("Hours > 0 (raw) — extreme right skew")
sns.histplot(np.log1p(h[h>0]), bins=60, ax=ax[1], color="#4C72B0"); ax[1].set_title("log(1+hours) — much tamer")
plt.tight_layout(); plt.show()

## 2.7 Univariate analysis — categorical features

In [ ]:
cat_cols = train.select_dtypes(include="object").columns
card = train[cat_cols].nunique().sort_values(ascending=False)
plt.figure(figsize=(9,6))
sns.barplot(x=card.head(15).values, y=card.head(15).index, color="#55A868")
plt.xlabel("number of distinct values"); plt.title("Categorical cardinality (top 15)")
plt.tight_layout(); plt.show()
print("A few columns have 1000s of levels -> one-hot would create thousands of columns.")

**Distribution of the main equipment group and size**

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16,4))
train[GROUP].value_counts().plot(kind="bar", ax=ax[0], color="#4C72B0"); ax[0].set_title("Equipment group")
order_size = ["Mini","Compact","Small","Medium","Large / Medium","Large"]
train["AssetScaleFactor"].value_counts().reindex(order_size).plot(kind="bar", ax=ax[1], color="#8172B3"); ax[1].set_title("Size (ordinal)")
train["UtilizationTier"].value_counts().reindex(["Low","Medium","High"]).plot(kind="bar", ax=ax[2], color="#C44E52"); ax[2].set_title("Utilization tier (ordinal)")
plt.tight_layout(); plt.show()

## 2.7 Missing-value analysis

**How much, and where?**

In [ ]:
miss = train.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
plt.figure(figsize=(8, max(4, 0.3*len(miss))))
sns.barplot(x=miss.values, y=miss.index, color="#4C72B0")
plt.xlabel("fraction missing"); plt.title("Missing fraction by column")
plt.tight_layout(); plt.show()
print(f"{len(miss)} of {train.shape[1]} columns have missing values")

**Types of missingness we see**

* Structural / MNAR — spec columns blank because the attribute doesn't apply to that machine type (most col*, Spec_*). 
* Placeholder-coded — ManufactureYear == 1001 and OperationalHoursMeter == 0/blank encode "unknown" as a fake value 
* Near-empty columns — col18, col19 are ~99.9% blank

## 2.8 Bivariate analysis — what moves the price?

**Numeric drivers: age and hours vs price**

In [ ]:
tmp = train.copy()
tmp["sale_year"] = pd.to_datetime(tmp[DATE]).dt.year
tmp["ym"] = tmp[YEAR_COL].where((tmp[YEAR_COL] > 1900) & (tmp[YEAR_COL] <= tmp["sale_year"]))
tmp["age"] = (tmp["sale_year"] - tmp["ym"]).clip(lower=0)

fig, ax = plt.subplots(1, 2, figsize=(14,4))
a = tmp[tmp["age"] <= 60]
a.groupby(a["age"].round())[TARGET].median().plot(marker="o", ax=ax[0], color="#C44E52")
ax[0].set_xlabel("age at sale (yrs)"); ax[0].set_ylabel("median price"); ax[0].set_title("Price falls with age")
hh = tmp[(tmp[HOURS] > 0) & (tmp[HOURS] < 30000)]
sns.regplot(data=hh.sample(4000, random_state=0), x=HOURS, y=TARGET,
            scatter_kws=dict(s=6, alpha=0.2), line_kws=dict(color="red"), ax=ax[1])
ax[1].set_title("Price vs usage hours (sampled)")
plt.tight_layout(); plt.show()

**Categorical drivers: price by group, size, and utilization**

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17,4))
for a, col, title in zip(ax, [GROUP, "AssetScaleFactor", "UtilizationTier"],
                         ["Equipment group", "Size", "Utilization"]):
    order = train.groupby(col)[TARGET].median().sort_values().index
    sns.boxplot(data=train, x=col, y=TARGET, order=order, showfliers=False, ax=a)
    a.set_title(f"Price by {title}"); a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

**Price over time (is there a market trend?)**

In [ ]:
ts = train.copy(); ts["ym"] = pd.to_datetime(ts[DATE]).dt.to_period("M").dt.to_timestamp()
ts.groupby("ym")[TARGET].median().plot(figsize=(12,3.5), color="#4C72B0")
plt.ylabel("median price"); plt.title("Median sale price over time")
plt.tight_layout(); plt.show()

## 2.10 Correlation & interaction effects

**Numeric correlation with log-price**

In [ ]:
cc = tmp.assign(logp=np.log1p(tmp[TARGET]),
                has_hours=(tmp[HOURS].fillna(0) > 0).astype(int))
numeric_for_corr = ["ym", "age", HOURS, "has_hours", "sale_year", "logp"]
corr = cc[numeric_for_corr].corr(numeric_only=True)
plt.figure(figsize=(6,5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation (engineered numeric vs log-price)")
plt.tight_layout(); plt.show()

**Interaction: does age affect price differently per equipment group?**

In [ ]:
inter = tmp[(tmp["age"] <= 40)].copy()
inter["age_bin"] = (inter["age"] // 5) * 5
piv = inter.pivot_table(index="age_bin", columns=GROUP, values=TARGET, aggfunc="median")
piv.plot(figsize=(11,4), marker="o")
plt.ylabel("median price"); plt.xlabel("age (5-yr bins)")
plt.title("Age vs price, split by equipment group (interaction)")
plt.legend(fontsize=8, ncol=2); plt.tight_layout(); plt.show()

## 2.11 Redundant & low-value columns

In [ ]:
# InventoryGroupCategory vs InventoryGroupDescription — is one a copy of the other?
rel = train.groupby("InventoryGroupCategory")["InventoryGroupDescription"].nunique()
print("Each InventoryGroupCategory maps to this many descriptions:")
print(rel.to_dict(), "-> perfectly redundant, keep only one")
print("\nNear-empty columns (>99% missing):",
      [c for c in train.columns if train[c].isna().mean() > 0.99])
constant = [c for c in train.columns if train[c].nunique(dropna=True) <= 1]
print("Constant columns:", constant if constant else "none")

## 2.12 EDA summary — key conclusions
**Data shape & quality**
1. 138,701 training rows × 50 columns; **no duplicate rows or IDs**.
2. 44 categorical + a few numeric columns; several categoricals reach **thousands of levels** (`ProductConfigID`, `Spec_FullDescriptor`, `Spec_BaseClass`).

**Target**
3. `TargetValue` ranges ~7.5k–142k, **right-skewed**; `log(1+price)` is near-symmetric → **model the log target** (matches RMSLE).

**Missing values & hidden placeholders**
4. Missingness is largely **structural** (spec doesn't apply) → treat "missing" as its own signal, don't drop rows.
5. **Hidden unknowns:** `ManufactureYear == 1001` (14.7k rows) and `OperationalHoursMeter == 0/blank` (~88k rows) → convert to NaN and add flags.
6. `col18`, `col19` are ~99.9% empty → **drop**.

**What drives price**
7. Strongest signals: **age**, **equipment group**, **size (AssetScaleFactor)**, and the **model/spec** categoricals; `ManufactureYear` (cleaned) correlates ≈0.30 with log-price.
8. **Age × group interaction** exists (different depreciation per machine type).
9. Mild **time trend** → calendar features worth adding.

**Encoding decisions**
10. `UtilizationTier` and `AssetScaleFactor` are **ordinal** (respect order); high-cardinality columns → **ordinal-encode + frequency features** (never one-hot); nominal columns → standard encoding. Scaling only matters for the linear baseline, not for tree models.

**Columns to drop:** identifiers (`TransactionID`, `AssetID`), near-empty (`col18`, `col19`), and one of the redundant `InventoryGroup*` pair.

*These conclusions directly define the preprocessing pipeline and feature-engineering function used in the next sections.*

## 2.13 Preprocessing and Feature engineering plan (derived directly from the EDA above)

### Part A — Feature Engineering Plan
*Creating new columns from existing ones to give the model better information.*

| New Feature | Built From | Why — Evidence From EDA |
|---|---|---|
| `log1p(TargetValue)` | TargetValue | Raw price is right-skewed (skew = +0.97). The competition metric is RMSLE which measures error in log space. Modelling log price directly optimises the right thing. (Section 2.4) |
| `sale_year` | TransactionDate | The time trend plot showed median price drifts over years. The model needs to know when the sale happened. (Section 2.8) |
| `sale_month` | TransactionDate | Auction prices show mild seasonality across months. (Section 2.8) |
| `sale_quarter` | TransactionDate | Groups months into 4 seasons — a coarser but stable time signal. (Section 2.8) |
| `sale_dow` | TransactionDate | Day of week of the auction — may reflect auction house patterns. (Section 2.8) |
| `sale_doy` | TransactionDate | Day of year — captures seasonal position more precisely than month. (Section 2.8) |
| `year_made_clean` | ManufactureYear | Raw ManufactureYear contains 14,719 rows with value 1001 — a placeholder for unknown. These are converted to NaN. The cleaned column contains only real years. (Section 2.5) |
| `age` | sale_year − year_made_clean | Age at the time of sale. The bivariate plot showed a strong, clean negative relationship with price — older machines sell for less. More directly useful than the raw year. (Section 2.8) |
| `has_hours` | OperationalHoursMeter | 40.8% of rows are missing hours AND 31,583 rows have hours recorded as exactly 0 — a second hidden unknown. A binary flag (1 = real reading exists, 0 = unknown) captures whether hours information is available at all. (Section 2.5, 2.7) |
| `hours_per_year` | hours ÷ age | Usage intensity — a machine with 10,000 hours over 5 years is more worn than one with 10,000 hours over 20 years. Raw hours alone does not capture this. |
| `*_freq` counts | ProductConfigID, Spec_FullDescriptor, Spec_BaseClass, FunctionalClassification, Spec_SubClass | These columns have thousands of unique levels. A frequency count tells the model how common or rare each model/spec is — common configurations tend to have more stable and predictable prices. (Section 2.6) |

---

### Part B — Preprocessing Plan
*Cleaning and transforming the data so the model can read it correctly. Applied identically to both train and test to avoid leakage.*

**Columns to drop — before anything else**

| Column | Reason |
|---|---|
| `TransactionID` | Unique identifier — carries no price information |
| `AssetID` | Individual machine identifier — not a learnable feature |
| `col18`, `col19` | 99.9% missing — essentially empty, no signal |
| `InventoryGroupCategory` | Perfectly redundant with InventoryGroupDescription — both encode the same 6 equipment groups, one as a code and one as a label. Keep only InventoryGroupDescription. (Section 2.10) |
| `TransactionDate` | Replaced entirely by the 5 calendar features above |
| `ManufactureYear` | Replaced by year_made_clean and age |
| `OperationalHoursMeter` | Replaced by hours, has_hours, and hours_per_year |

---

**Handling missing values — numeric columns**

Missing values in numeric columns are filled with the **median** of that column.

Why median and not mean?
Several numeric columns (like hours) are heavily right-skewed with extreme outliers. The mean gets pulled toward those outliers — for example, one machine with 1,729,600 hours would push the mean far above what a typical machine looks like. The median is the middle value and is not affected by extremes.

A **missing-indicator column** is also added for each numeric column that had missing values. This is a 0/1 flag that tells the model whether that value was originally present or was filled in. This is important because missingness itself is informative — a machine with no hours reading is different from one with a known reading.

---

**Handling missing values — categorical columns**

Missing values in categorical columns are filled with a literal string `"__NA__"`.

Why not drop them or use the most frequent value?
As the missingness heatmap showed (Section 2.7), missing spec columns are structural — the attribute simply does not apply to that machine type. Replacing with a constant `"__NA__"` keeps that information visible to the model as its own category level. The model can learn that `"__NA__"` in a spec column behaves differently from any real value.

---

**Encoding categorical columns**

Categorical columns contain text — the model cannot read text directly. They need to be converted to numbers.

Two types of encoding are used:

*Ordinal encoding for all categorical columns:*
Each unique category level is assigned an integer. For example Backhoe Loaders = 0, Motor Graders = 1, Track Excavators = 2, and so on. Tree-based models (Random Forest, LightGBM, XGBoost, CatBoost) do not care about the numeric order — they find the right split points themselves.

Any category level seen in test but not in train is assigned -1 (unknown value). This is handled automatically.

*Why not one-hot encoding?*
One-hot encoding creates one new column per unique category level. Spec_FullDescriptor has 3,505 unique levels — one-hot would add 3,505 columns. With multiple high-cardinality columns the dataset would explode to tens of thousands of columns, making training very slow and the model prone to overfitting.

---

**Scaling numeric columns**

Scaling (standardising numeric values to a common range) is applied only for the **Ridge regression** model. Tree-based models do not need scaling — they split on thresholds, not on the actual magnitude of values. Applying scaling to tree models makes no difference to their predictions.

---

**Pipeline — why everything is wrapped in a pipeline**

All the above steps — imputing, encoding, scaling — are wrapped inside a scikit-learn Pipeline and ColumnTransformer.


### Summary — what changes between raw data and model input

```
Raw data (50 columns)
        ↓
Drop 7 columns (IDs, near-empty, redundant, replaced columns)
        ↓
Add 11 new features (calendar, age, hours flags, frequency counts)
        ↓
Fill missing numerics with median + add missing-indicator flags
        ↓
Fill missing categoricals with "NA"
        ↓
Encode all categoricals as integers (ordinal encoding)
        ↓
Scale numerics (for Ridge only)
        ↓
Model input: clean numeric matrix, no missing values, no text
```



# 3. Feature Engineering


In [ ]:

#step 1: drop columns that carry no useful information 

DROP_COLS = [
    "TransactionID",          # just a serial number, not a feature
    "AssetID",                # individual machine id — too specific to generalise
    "InventoryGroupCategory", # exact duplicate of InventoryGroupDescription (EDA 2.10)
    "col18",                  # 99.95% missing — no usable signal (EDA 2.7)
    "col19",                  # 99.95% missing — no usable signal (EDA 2.7)
]



## 3.1 Date features from TransactionDate

The EDA showed a mild time trend in prices across years and months (Section 2.8).
The model cannot read a date string — we extract the useful parts as numbers.
After extraction, the raw date column is dropped since all its information
is now captured in the new columns.
```

In [ ]:

def extract_date_features(df):
    d = pd.to_datetime(df["TransactionDate"], errors="coerce")
    df["sale_year"]    = d.dt.year        # which year was the auction
    df["sale_month"]   = d.dt.month       # which month (1-12)
    df["sale_quarter"] = d.dt.quarter     # Q1/Q2/Q3/Q4
    df["sale_dow"]     = d.dt.dayofweek   # day of week (0=Monday, 6=Sunday)
    df["sale_doy"]     = d.dt.dayofyear   # day of year (1-365)
    return df



## 3.2 Cleaning ManufactureYear and building Age

The EDA found 14,719 rows where ManufactureYear = 1001 — a placeholder
meaning "year unknown" (Section 2.5). Using 1001 directly would give the
model nonsense ages like 1004 years old.

We fix this in two steps:
1. Replace any year below 1900 with NaN (honest missing)
2. Build age = sale year minus manufacture year

Age is a much more useful feature than the raw year because what matters
to price is how old the machine was when it was sold, not what year it 
was built in isolation.

We also clip age at 0 — if the data has any error where manufacture year
is after sale year, we do not want a negative age.


In [ ]:

def build_age_feature(df):
    ym = pd.to_numeric(df["ManufactureYear"], errors="coerce")
    
    # remove impossible years — anything below 1900 is a placeholder
    ym = ym.where((ym > 1900) & (ym <= df["sale_year"]))
    
    df["year_made_clean"] = ym
    df["age"] = (df["sale_year"] - ym).clip(lower=0)
    
    return df



## 3.3 Usage hours features

OperationalHoursMeter had two problems found in EDA (Section 2.5, 2.7):
- 40.8% of rows are genuinely missing
- Another 31,583 rows have hours recorded as exactly 0 — a second hidden unknown

We build three things from this column:

1. hours — the clean numeric reading (zeros kept as NaN so imputer handles them)
2. has_hours — a 0/1 flag: did this machine have a real hours reading?
   This is important because a machine with no reading is fundamentally
   different from one with a known reading
3. hours_per_year — usage intensity (how hard was the machine worked each year)
   A machine with 10,000 hours over 5 years is far more worn than one with
   10,000 hours over 20 years. Raw hours alone does not capture this.


In [ ]:

def build_hours_features(df):
    mh = pd.to_numeric(df["OperationalHoursMeter"], errors="coerce")
    
    # treat 0 as unknown — a machine cannot have zero lifetime hours
    mh = mh.where(mh > 0)
    
    df["hours"]         = mh
    df["has_hours"]     = mh.notna().astype(int)  # 1 if real reading, 0 if unknown
    df["hours_per_year"] = mh / df["age"].replace(0, np.nan)  # usage intensity
    
    return df



## 3.4 Frequency features for high-cardinality columns

Some columns like Spec_FullDescriptor and ProductConfigID have thousands of
unique values (Section 2.6). We cannot one-hot encode them and ordinal encoding
gives them an arbitrary order.

A frequency count is a simple but effective solution — it tells the model
how common or rare each configuration is. Common configurations appear many
times in the training data so the model has seen many price examples for them.
Rare configurations have very few examples and tend to behave differently.

Important: frequency maps are learned from training data only, then applied
to both train and test. This is to prevent leakage — the test set cannot
influence what counts we assign.


In [ ]:

FREQ_COLS = [
    "ProductConfigID",
    "Spec_FullDescriptor",
    "Spec_BaseClass",
    "FunctionalClassification",
    "Spec_SubClass",
]

def fit_frequency_maps(train_df):
    # learn counts from training data only
    maps = {}
    for col in FREQ_COLS:
        if col in train_df.columns:
            maps[col] = train_df[col].astype("string").value_counts()
    return maps

def apply_frequency_maps(df, maps):
    df = df.copy()
    for col, counts in maps.items():
        # unseen values in test get 0 (they were never in training)
        df[col + "_freq"] = df[col].astype("string").map(counts).fillna(0)
    return df

# fit on train only — then apply to both
freq_maps = fit_frequency_maps(train)
train = apply_frequency_maps(train, freq_maps)
test  = apply_frequency_maps(test,  freq_maps)



## 3.5 Drop raw columns that have been fully replaced

After building all the new features, the original raw columns they came from
are no longer needed. Keeping them would confuse the model — for example,
ManufactureYear = 1001 should not sit alongside a clean age feature.

TransactionDate is also dropped here since all its information now lives
in the five calendar columns.


In [ ]:

REPLACE_DROP = [
    "TransactionDate",      # replaced by sale_year, sale_month, sale_quarter, sale_dow, sale_doy
    "ManufactureYear",      # replaced by year_made_clean and age
    "OperationalHoursMeter",# replaced by hours, has_hours, hours_per_year
]



## 3.6 The complete add_features function

All the steps above are combined into one reusable function.
Calling it on train and test gives both datasets the same new columns
with no possibility of one contaminating the other.


In [ ]:

def add_features(df):
    df = df.copy()
    
    # date features first — sale_year needed for age calculation below
    df = extract_date_features(df)
    
    # clean manufacture year and build age
    df = build_age_feature(df)
    
    # usage hours features
    df = build_hours_features(df)
    
    # drop columns no longer needed
    cols_to_drop = DROP_COLS + REPLACE_DROP
    cols_to_drop = [c for c in cols_to_drop if c in df.columns]
    df = df.drop(columns=cols_to_drop)
    
    return df


# apply identically to both train and test
train_fe = add_features(train)
test_fe  = add_features(test)

print("Original columns:", train.shape[1])
print("After engineering:", train_fe.shape[1])
print()
print("New columns added:")
new_cols = [c for c in train_fe.columns if c not in train.columns]
print(new_cols)
print()
print("Columns removed:")
removed = [c for c in train.columns if c not in train_fe.columns]
print(removed)



## 3.7 Quick check — does the engineered data look right?

Before moving to preprocessing, a few sanity checks to make sure
nothing went wrong in the transformations.


In [ ]:

# age should be between 0 and ~60, with some NaN for unknown years
print("age — missing: {:.1f}%  |  min: {:.0f}  |  median: {:.0f}  |  max: {:.0f}".format(
    100 * train_fe["age"].isna().mean(),
    train_fe["age"].min(),
    train_fe["age"].median(),
    train_fe["age"].max()))

# has_hours should be 0 or 1 only
print("has_hours values:", train_fe["has_hours"].value_counts().to_dict())

# sale_year should be between 1989 and 2013
print("sale_year range:", train_fe["sale_year"].min(), "to", train_fe["sale_year"].max())

# check no target column was accidentally dropped
assert "TargetValue" in train_fe.columns, "TargetValue was accidentally dropped"

# check train and test have the same columns (except TargetValue)
train_cols = set(train_fe.columns) - {"TargetValue"}
test_cols  = set(test_fe.columns)
if train_cols == test_cols:
    print("Train and test columns match perfectly")
else:
    print("MISMATCH — in train not test:", train_cols - test_cols)
    print("MISMATCH — in test not train:", test_cols - train_cols)


# 4. Preprocessing

## 4.1 Identifying column types

After feature engineering we have three types of columns that each need
different treatment.

In [ ]:
# columns we built during feature engineering
ENGINEERED_NUM = [
    "sale_year", "sale_month", "sale_quarter", "sale_dow", "sale_doy",
    "year_made_clean", "age", "hours", "has_hours", "hours_per_year",
    "ProductConfigID_freq", "Spec_FullDescriptor_freq",
    "Spec_BaseClass_freq", "FunctionalClassification_freq", "Spec_SubClass_freq",
]

# ordinal — categories with a natural order (from EDA section 2.2)
ORDINAL_COLS  = ["UtilizationTier", "AssetScaleFactor"]

# the order matters for ordinal encoding — must go low to high
ORDINAL_ORDER = [
    ["Low", "Medium", "High"],                                          # UtilizationTier
    ["Mini", "Compact", "Small", "Medium", "Large / Medium", "Large"],  # AssetScaleFactor
]

# set up X and y
TARGET = "TargetValue"
y = np.log1p(train_fe[TARGET].values)   # model log price — matches RMSLE metric
X = train_fe.drop(columns=[TARGET])
X_test = test_fe.reindex(columns=X.columns)  # align test columns to train

# auto-detect column groups
num_cols = X.select_dtypes(include="number").columns.tolist()
ord_cols = [c for c in ORDINAL_COLS if c in X.columns]
cat_cols = [c for c in X.columns if c not in num_cols and c not in ord_cols]

print(f"Numeric  : {len(num_cols)} columns")
print(f"Ordinal  : {len(ord_cols)} columns")
print(f"Categorical: {len(cat_cols)} columns")

## 4.2 Handling missing values in numeric columns

Numeric columns are imputed with the median.

Why median?
Several columns like hours and hours_per_year are heavily right-skewed
with extreme values. For example, one machine had 1,729,600 hours recorded.
The mean gets pulled toward this extreme value and would give an unrealistic
fill for typical missing rows. The median is the middle value and is not
affected by extremes.

add_indicator=True adds an extra 0/1 column for each numeric column
that had missing values. This flag tells the model whether the value was
originally present or was filled in — because missingness itself can be
informative (a machine with no age record may behave differently from
one with a known age).

In [ ]:
numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
])

## 4.3 Handling missing values in ordinal columns

UtilizationTier is 63.6% missing. AssetScaleFactor is 29.7% missing.

For these we fill with the string "__NA__" before encoding.
This gives the model its own category level for "unknown" — 
it can learn that machines with no size recorded behave
differently from Small or Large machines.

The OrdinalEncoder then converts the levels to numbers in the 
correct order: Low=0, Medium=1, High=2 for UtilizationTier and
Mini=0 through Large=5 for AssetScaleFactor.

handle_unknown="use_encoded_value" with unknown_value=-1 means
any category level seen in test but not in train gets assigned -1.
This prevents the pipeline from crashing on unseen values.

In [ ]:
ordinal_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="__NA__")),
    ("encode", OrdinalEncoder(
        categories=ORDINAL_ORDER,
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )),
])

## 4.4 Handling missing values in categorical columns

The remaining 38 categorical columns — equipment specs, region, cabin type,
vendor, drivetrain and so on — all get the same treatment.

Missing values are filled with "__NA__" — a literal string that becomes
its own category level. The model can learn that "__NA__" in a spec column
is not random — it usually means that spec does not apply to that machine type.

OrdinalEncoder converts every unique string to an integer.
The order is arbitrary for nominal categories — that is fine because
tree models (Random Forest, LightGBM, XGBoost, CatBoost) find their own 
split points and do not care about the numeric order assigned.

In [ ]:
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="__NA__")),
    ("encode", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )),
])

## 4.5 Combining everything into one ColumnTransformer

ColumnTransformer applies a different pipeline to each group of columns
and then stitches the results back together into one matrix.

All three are handled simultaneously in one step.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("ord", ordinal_pipe, ord_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"   # any column not listed above is dropped
)

## 4.6 Validation split

We split the training data into two parts before fitting anything:
- 80% for training the model
- 20% for validating it — the model never sees this during training

This gives an honest estimate of how well the model will perform
on the actual test set.

random_state=42 makes the split reproducible — running the notebook
again gives the same split every time.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training rows  :", X_train.shape[0])
print("Validation rows:", X_val.shape[0])
print("Features       :", X_train.shape[1])

## 4.7 The RMSLE metric function

The competition measures error using RMSLE — Root Mean Squared Log Error.
Since we already trained on log(price), computing RMSLE is straightforward:
convert predictions back to price with expm1, then compare with actual prices.

We define this once here and use it consistently to evaluate every model.

In [ ]:
def rmsle(actual_price, predicted_price):
    # clip predictions at 0 — price cannot be negative
    predicted_price = np.clip(predicted_price, 0, None)
    return np.sqrt(
        np.mean(
            (np.log1p(predicted_price) - np.log1p(actual_price)) ** 2
        )
    )

# actual prices for the validation set (converted back from log)
price_val = np.expm1(y_val)

## 4.8 Quick check — does the preprocessor work correctly?

Before building model, we verify the pipeline runs without errors
and produces the expected output shape.

In [ ]:
# fit on training data only — never on validation or test
preprocessor.fit(X_train)

# transform validation to check shape
val_transformed = preprocessor.transform(X_val)
print("Input shape (train) :", X_train.shape)
print("Output shape (train):", preprocessor.transform(X_train).shape)
print()
print("Extra columns from add_indicator (missing flags):",
      val_transformed.shape[1] - X_train.shape[1])
print()

# confirm no NaN values remain after preprocessing
import numpy as np
nan_count = np.isnan(preprocessor.transform(X_train)).sum()
print("NaN values remaining after preprocessing:", nan_count)
print("All good." if nan_count == 0 else "WARNING — NaN values remain")

**Note on scaling**

Scaling (StandardScaler — converts values to mean=0, std=1) is NOT applied
in the shared preprocessor above.

Why?
Tree models — Random Forest, LightGBM, XGBoost, CatBoost — split data on
thresholds. Whether a column goes from 0 to 60 (age) or 0 to 1,700,000 (hours)
makes no difference to a tree. It just finds the best split point within
whatever range exists.

Scaling only matters for the Ridge regression model, which uses distances
and dot products that are sensitive to column magnitudes.

So scaling is added only inside the Ridge model pipeline — shown in the
model comparison section — not here in the shared preprocessor.

# 5. Model building


We train five models from three different families:

- Linear family   → Ridge Regression
- Bagging family  → Random Forest  
- Boosting family → LightGBM, XGBoost, CatBoost


All five models:
- receive exactly the same preprocessed features (fair comparison)
- are evaluated on the same validation set (consistent measurement)
- are measured using RMSLE (the actual competition metric)

Target variable reminder: we trained on log(1+price), so predictions
are converted back with expm1() before computing RMSLE.

In [ ]:
## 5.2 — Helper function for consistent evaluation

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

results = {}   # stores results for all models — used in comparison later

def evaluate(model, name, scale=False):
    """
    Wraps the model in a pipeline with preprocessing,
    trains on X_train, evaluates on X_val, stores the result.
    
    scale=True only for Ridge — tree models do not need scaling.
    """
    steps = [("prep", preprocessor)]
    if scale:
        steps.append(("scale", StandardScaler(with_mean=False)))
    steps.append(("model", model))
    
    pipe = Pipeline(steps)
    
    t = time.time()
    pipe.fit(X_train, y_train)
    elapsed = round(time.time() - t, 1)
    
    # predict in log space, convert back to price for RMSLE
    pred_price = np.expm1(pipe.predict(X_val))
    score = rmsle(price_val, pred_price)
    
    results[name] = {
        "rmsle"  : round(score, 4),
        "seconds": elapsed,
        "pipe"   : pipe
    }
    
    print(f"{name:20s}  val RMSLE = {score:.4f}   ({elapsed}s)")
    return pipe

## 5.1 Model 1: Ridge Regression


In [ ]:
ridge_pipe = evaluate(
    Ridge(alpha=10),
    name="Ridge",
    scale=True       # only model that needs scaling
)

## 5.2 Model 2: Random Forest

Random Forest trains many decision trees, each on a random sample
of rows and features, then averages their predictions.


In [ ]:
rf_pipe = evaluate(
    RandomForestRegressor(
        n_estimators=80,
        min_samples_leaf=3,
        max_features=0.5,
        n_jobs=-1,
        random_state=42
    ),
    name="RandomForest"
)

## 5.3 Model 3: LightGBM

LightGBM is a gradient boosting model. It builds trees sequentially —
each new tree learns from the mistakes of all previous trees.

In [ ]:
lgbm_pipe = evaluate(
    LGBMRegressor(
        n_estimators=800,
        learning_rate=0.03,
        num_leaves=127,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1
    ),
    name="LightGBM"
)

## 5.4 Model 4: XGBoost


In [ ]:
xgb_pipe = evaluate(
    XGBRegressor(
        n_estimators=800,
        learning_rate=0.03,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=42,
        verbosity=0
    ),
    name="XGBoost"
)

## 5.5 Model 5: CatBoost

In [ ]:
cat_pipe = evaluate(
    CatBoostRegressor(
        iterations=600,
        learning_rate=0.05,
        depth=8,
        random_state=42,
        verbose=0
    ),
    name="CatBoost"
)

# 6. Model Evaluation: 

## 6.1 Comparing All Five Models

Now that all five models are trained, we compare them side by side
using the same validation set and the same metric (RMSLE).

In [ ]:
# Comparison table

comparison = pd.DataFrame({
    name: {"Validation RMSLE": v["rmsle"], "Training time (s)": v["seconds"]}
    for name, v in results.items()
}).T.sort_values("Validation RMSLE")

print("Model comparison — sorted by RMSLE (lower is better)")
print()
display(comparison)

In [ ]:
# Comparison bar chart

plt.figure(figsize=(9, 4))
bars = plt.bar(
    comparison.index,
    comparison["Validation RMSLE"],
    color=["#55A868" if v < 0.20 else "#4C72B0" for v in comparison["Validation RMSLE"]]
)
plt.axhline(0.20, color="red", linestyle="--", linewidth=1.5, label="0.20 cutoff")
plt.ylabel("Validation RMSLE")
plt.title("Baseline model comparison (lower is better)")
plt.legend()

# add score labels on top of each bar
for bar, val in zip(bars, comparison["Validation RMSLE"]):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.003,
             f"{val:.4f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

## 6.2 Observations from comparison

- Ridge (linear) scores ~0.40 — confirms the price relationships 
  are non-linear. A straight line is not enough.

- Random Forest scores ~0.21 — a big jump from Ridge. 
  Non-linear tree splits capture the complex interactions.

- LightGBM and XGBoost score ~0.207–0.208 — best among baselines.
  Sequential boosting squeezes out more signal than bagging.

- CatBoost scores ~0.22 with default settings — still strong but
  needs more iterations to compete with LightGBM/XGBoost.



# 7. Model Inspection: What Did the Model Learn?

Before tuning, we need to check whether the model learned
sensible things from the data. 

## 7.1 Feature importance from LightGBM

In [ ]:


best_lgbm = results["LightGBM"]["pipe"]
lgbm_model = best_lgbm.named_steps["model"]

# get feature names after preprocessing
# (preprocessing adds extra columns for missing indicators)
feature_names = best_lgbm.named_steps["prep"].get_feature_names_out()

imp = pd.Series(
    lgbm_model.feature_importances_,
    index=feature_names
).sort_values(ascending=False).head(20)

# clean up the prefixes sklearn adds (num__, cat__, etc.)
imp.index = [name.split("__")[-1] for name in imp.index]

plt.figure(figsize=(8, 6))
sns.barplot(x=imp.values, y=imp.index, color="#4C72B0")
plt.xlabel("Feature importance (gain)")
plt.title("Top 20 features — LightGBM")
plt.tight_layout()
plt.show()

# 8. Model Inference: Where Does the Model Go Wrong?

Residual analysis shows us the pattern of errors. If errors are
random (no pattern), the model is working well. If there is a clear
pattern in the errors, the model is missing something systematic.

## 8.1 Predicted vs actual plot

In [ ]:

val_pred_price = np.expm1(best_lgbm.predict(X_val))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))

# predicted vs actual
ax[0].scatter(
    np.log1p(price_val),
    np.log1p(val_pred_price),
    s=4, alpha=0.15, color="#4C72B0"
)
lims = [np.log1p(price_val).min(), np.log1p(price_val).max()]
ax[0].plot(lims, lims, "r--", linewidth=1)
ax[0].set_xlabel("Actual log price")
ax[0].set_ylabel("Predicted log price")
ax[0].set_title("Predicted vs actual (perfect model = diagonal line)")

# residuals
residuals = np.log1p(val_pred_price) - np.log1p(price_val)
sns.histplot(residuals, bins=60, ax=ax[1], color="#C44E52", kde=True)
ax[1].axvline(0, color="black", linestyle="--", linewidth=1)
ax[1].set_xlabel("Residual (predicted − actual) in log space")
ax[1].set_title(f"Residual distribution  (std = {residuals.std():.3f})")

plt.tight_layout()
plt.show()

print("Mean residual  : {:.4f}  (close to 0 = unbiased)".format(residuals.mean()))
print("Std of residual: {:.4f}  (smaller = better)".format(residuals.std()))
print("Max over-prediction : {:.4f}".format(residuals.max()))
print("Max under-prediction: {:.4f}".format(residuals.min()))

# 9. Hyperparameter tuning

Which hyperparameters we tune and what each one does

We focus on LightGBM because it gave the best baseline score.
Here are the seven parameters we search over, and what each controls:

**n_estimators** — number of trees in the sequence
- We search: [600, 900, 1200]


**learning_rate** — how much each new tree corrects the previous error
- We search: [0.02, 0.03, 0.05]


**num_leaves** — maximum leaves per tree, controls tree complexity
- We search: [63, 127, 255]


**min_child_samples** — minimum rows required in each leaf
- We search: [20, 40, 80]


**subsample** — fraction of rows used to build each tree
- We search: [0.7, 0.8, 0.9]

**colsample_bytree** — fraction of features used for each tree
- We search: [0.6, 0.8, 1.0]

**reg_lambda** — L2 regularization strength
- We search: [0.0, 1.0, 5.0]

**Why RandomizedSearchCV, not Grid Search?**

RandomizedSearchCV tests only a random subset of hyperparameter combinations instead of trying every possible one, making it much faster than Grid Search.
With n_iter=15 and cv=3, it evaluates 15 × 3 = 45 model fits instead of thousands.
3-fold cross-validation trains the model three times on different train-validation splits and averages the performance for a more reliable estimate.
The scoring metric neg_root_mean_squared_error measures RMSE (RMSLE for a log-transformed target), with the negative sign used because scikit-learn maximises scores internally.

## 9.1 Tuning LightGBM with RandomizedSearchCV

In [ ]:


from sklearn.model_selection import RandomizedSearchCV

lgbm_param_grid = {
    "model__n_estimators"      : [600, 900, 1200],
    "model__learning_rate"     : [0.02, 0.03, 0.05],
    "model__num_leaves"        : [63, 127, 255],
    "model__min_child_samples" : [20, 40, 80],
    "model__subsample"         : [0.7, 0.8, 0.9],
    "model__colsample_bytree"  : [0.6, 0.8, 1.0],
    "model__reg_lambda"        : [0.0, 1.0, 5.0],
}

lgbm_search = RandomizedSearchCV(
    Pipeline([
        ("prep",  preprocessor),
        ("model", LGBMRegressor(random_state=42, verbose=-1))
    ]),
    param_distributions = lgbm_param_grid,
    n_iter     = 15,     # try 15 random combinations
    cv         = 3,      # 3-fold cross-validation each time
    scoring    = "neg_root_mean_squared_error",  # = -RMSLE
    random_state = 42,   # reproducible random sampling
    n_jobs     = -1,     # use all available CPU cores
    verbose    = 1       # print progress
)

print("Starting LightGBM hyperparameter search...")
print("This will try 15 combinations × 3 folds = 45 model fits")
print("Expected time on Kaggle CPU: 15-25 minutes")
print()

lgbm_search.fit(X_train, y_train)

# evaluate best model on our held-out validation set
lgbm_tuned_pred  = np.expm1(lgbm_search.predict(X_val))
lgbm_tuned_rmsle = rmsle(price_val, lgbm_tuned_pred)

print()
print("Best cross-validation RMSLE : {:.4f}".format(-lgbm_search.best_score_))
print("Validation RMSLE            : {:.4f}".format(lgbm_tuned_rmsle))
print()
print("Best parameters found:")
best_params = {k.replace("model__", ""): v for k, v in lgbm_search.best_params_.items()}
for param, value in best_params.items():
    print(f"  {param:25s} = {value}")

** Understanding the cross-validation score vs validation score**

there are two scores printed above:
- Best cross-validation RMSLE — average across 3 folds on training data
- Validation RMSLE — on the 20% held-out set we never touched during search

These two numbers should be close to each other.
If CV score is much better than validation score, the model overfit
to the training data during search (unlikely with regularization).
If they match closely, the search found settings that genuinely generalise.

## 9.2 Tuning Random Forest (lighter search)

In [ ]:


# Random Forest is slower to train so we do a smaller search

rf_param_grid = {
    "model__n_estimators"   : [80, 120, 160],
    "model__min_samples_leaf": [1, 2, 3],
    "model__max_features"   : [0.3, 0.5, 0.7],
}

rf_search = RandomizedSearchCV(
    Pipeline([
        ("prep",  preprocessor),
        ("model", RandomForestRegressor(n_jobs=-1, random_state=42))
    ]),
    param_distributions = rf_param_grid,
    n_iter       = 8,
    cv           = 3,
    scoring      = "neg_root_mean_squared_error",
    random_state = 42,
    n_jobs       = -1,
    verbose      = 0
)

rf_search.fit(X_train, y_train)
rf_tuned_rmsle = rmsle(price_val, np.expm1(rf_search.predict(X_val)))

print("Random Forest tuned RMSLE: {:.4f}".format(rf_tuned_rmsle))
print("Best RF parameters:")
for k, v in rf_search.best_params_.items():
    print(f"  {k.replace('model__',''):20s} = {v}")

**Why we do not tune Ridge**

Ridge only has one meaningful hyperparameter — alpha (the penalty strength).
It was already tested with alpha=10 in the baseline.

More importantly, Ridge scored 0.40 on validation, roughly double the
error of any boosting model.

## 9.3 Tuning results: search history

In [ ]:


# how did scores change across the 15 combinations tried?
cv_results = pd.DataFrame(lgbm_search.cv_results_)
cv_results = cv_results.sort_values("rank_test_score").reset_index(drop=True)

plt.figure(figsize=(10, 4))
plt.plot(
    range(1, len(cv_results) + 1),
    -cv_results["mean_test_score"],
    marker="o", color="#4C72B0", markersize=5
)
plt.xlabel("Combination number (sorted best to worst)")
plt.ylabel("Cross-validation RMSLE")
plt.title("Search results — each dot is one hyperparameter combination tried")
plt.axhline(
    -cv_results["mean_test_score"].min(),
    color="red", linestyle="--", linewidth=1,
    label=f"Best: {-cv_results['mean_test_score'].min():.4f}"
)
plt.legend()
plt.tight_layout()
plt.show()

## 9.4 Full tuned model comparison

In [ ]:


tuned_results = {
    "Ridge (baseline)"       : results["Ridge"]["rmsle"],
    "RandomForest (baseline)": results["RandomForest"]["rmsle"],
    "LightGBM (baseline)"    : results["LightGBM"]["rmsle"],
    "XGBoost (baseline)"     : results["XGBoost"]["rmsle"],
    "CatBoost (baseline)"    : results["CatBoost"]["rmsle"],
    "RandomForest (tuned)"   : rf_tuned_rmsle,
    "LightGBM (tuned)"       : lgbm_tuned_rmsle,
}

cmp = pd.Series(tuned_results).sort_values()
display(cmp.round(4).to_frame("Validation RMSLE"))

plt.figure(figsize=(10, 5))
colors = ["#55A868" if v < 0.20 else "#4C72B0" for v in cmp.values]
bars = plt.bar(cmp.index, cmp.values, color=colors)
plt.axhline(0.20, color="red", linestyle="--", linewidth=1.5, label="0.20 cutoff")
plt.xticks(rotation=20, ha="right")
plt.ylabel("Validation RMSLE")
plt.title("Baseline vs tuned — all models compared")
plt.legend()
for bar, val in zip(bars, cmp.values):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.003,
             f"{val:.4f}", ha="center", fontsize=8)
plt.tight_layout()
plt.show()

# 10. Final model: LightGBM with native categorical handling + seed ensemble

**Why native categorical handling helps**

In the preprocessing pipeline, categorical columns were encoded as
arbitrary integers:
  Backhoe Loaders  = 0
  Motor Graders    = 1
  Track Excavators = 2
  ...

The order is random — LightGBM has to figure out on its own that
categories 0 and 3 might behave similarly in terms of price.

With native categorical handling, LightGBM reads the column as a
true category. At each split it searches for the optimal grouping
of category levels — for example:
  {Wheel Loader, Track Type Tractor} vs {all others}

This is especially powerful for high-cardinality columns like
Spec_FullDescriptor (3,505 levels) and Spec_BaseClass (1,249 levels)
where finding the right grouping matters a lot

## 10.1 Prepare native categorical views of the data

In [ ]:

# all categorical columns — both nominal and ordinal
all_cat_cols = cat_cols + ord_cols

# cast to pandas category dtype
# LightGBM reads this dtype as a signal to use native categorical splits
X_native      = X.copy()
X_test_native = X_test.copy()

for c in all_cat_cols:
    X_native[c] = X_native[c].astype("category")
    # align test categories to train — unseen test levels become NaN
    X_test_native[c] = pd.Categorical(
        X_test_native[c].astype("string"),
        categories=X_native[c].cat.categories
    )

print("Categorical columns set to native handling:", len(all_cat_cols))
print("Numeric columns still imputed normally   :", len(num_cols))

## 10.2 Impute numeric columns (categoricals handled natively by LightGBM)

In [ ]:


from sklearn.impute import SimpleImputer

# fit imputer on training data only — same leakage-prevention rule as before
num_imputer = SimpleImputer(strategy="median")

# validation split of the native categorical data
X_tr_nat, X_va_nat, y_tr_nat, y_va_nat = train_test_split(
    X_native, y, test_size=0.2, random_state=42
)
price_va_nat = np.expm1(y_va_nat)

# impute numeric columns
X_tr_nat = X_tr_nat.copy()
X_va_nat = X_va_nat.copy()
X_tr_nat[num_cols] = num_imputer.fit_transform(X_tr_nat[num_cols])
X_va_nat[num_cols] = num_imputer.transform(X_va_nat[num_cols])

print("Numeric imputation done — no NaN values remain in numeric columns")
print("Categorical columns left as-is for LightGBM native handling")

## 10.3 Finding the optimal number of trees with early stopping

Instead of guessing how many trees to use, we use early stopping.

The model trains on the training split and checks its score on the
validation split after every tree. If the score does not improve for
120 consecutive trees, training stops automatically.

This finds the exact number of trees that gives the best validation
score without overfitting.


In [ ]:


from lightgbm import LGBMRegressor, early_stopping

FINAL_PARAMS = dict(
    learning_rate     = 0.02,
    num_leaves        = 255,
    subsample         = 0.8,
    colsample_bytree  = 0.7,
    min_child_samples = 30,
    reg_lambda        = 1.0,
    verbose           = -1
)

print("Training probe model to find optimal number of trees...")
print("Model will stop automatically when validation score stops improving")
print()

probe_model = LGBMRegressor(
    n_estimators = 6000,     # maximum allowed — early stopping will reduce this
    random_state = 42,
    **FINAL_PARAMS
)

probe_model.fit(
    X_tr_nat, y_tr_nat,
    eval_set   = [(X_va_nat, y_va_nat)],
    callbacks  = [early_stopping(stopping_rounds=120, verbose=False)]
)

BEST_ITER = probe_model.best_iteration_
probe_score = rmsle(price_va_nat, np.expm1(probe_model.predict(X_va_nat)))

print(f"Optimal number of trees : {BEST_ITER}")
print(f"Validation RMSLE        : {probe_score:.4f}")

## 10.4 Seed ensemble: why averaging models helps

Even with the same data and same hyperparameters, two LightGBM models
trained with different random seeds produce slightly different trees.

The randomness comes from:
- subsample=0.8 — each tree samples 80% of rows randomly
- colsample_bytree=0.7 — each tree samples 70% of features randomly
- the random seed controls which 80% and 70% are chosen

Different random selections → different trees → different errors.

When we average predictions from three differently-seeded models,
their errors partly cancel out, one model over predicts where another
under predicts, and the average lands closer to the truth.


In [ ]:


SEEDS = [42, 1, 7]

val_predictions  = np.zeros(len(y_va_nat))
print("Seed ensemble — validation scores:")
for seed in SEEDS:
    m = LGBMRegressor(n_estimators=BEST_ITER, random_state=seed, **FINAL_PARAMS)
    m.fit(X_tr_nat, y_tr_nat)
    pred = m.predict(X_va_nat)
    val_predictions += pred / len(SEEDS)
    single_score = rmsle(price_va_nat, np.expm1(pred))
    print(f"  seed={seed}  RMSLE={single_score:.4f}")

ensemble_score = rmsle(price_va_nat, np.expm1(val_predictions))
print()
print(f"Single model RMSLE  : {probe_score:.4f}")
print(f"Ensemble RMSLE      : {ensemble_score:.4f}")
print(f"Improvement         : {probe_score - ensemble_score:.4f}")

## 10.5 Refit on all training data + predict test set

Until now the model was trained on 80% of the data (110,960 rows)
and validated on the remaining 20% (27,741 rows).

For the final submission we refit using ALL 138,701 training rows.
The number of trees stays fixed at BEST_ITER (found by early stopping above).
We do not use early stopping here because there is no held-out set
to monitor — we use the tree count we already validated.

In [ ]:

# impute numeric columns using ALL training data this time
final_imputer = SimpleImputer(strategy="median")

X_all_nat = X_native.copy()
X_all_nat[num_cols] = final_imputer.fit_transform(X_all_nat[num_cols])

# apply same imputer to test
X_test_final = X_test_native.copy()
X_test_final[num_cols] = final_imputer.transform(X_test_final[num_cols])

# seed ensemble — refit each seed on ALL data, predict test
test_predictions = np.zeros(len(X_test_final))

print("Refitting on all training data and predicting test set...")
for seed in SEEDS:
    m = LGBMRegressor(n_estimators=BEST_ITER, random_state=seed, **FINAL_PARAMS)
    m.fit(X_all_nat, y)
    test_predictions += m.predict(X_test_final) / len(SEEDS)
    print(f"  seed={seed} done")

# convert from log space back to price
final_prices = np.clip(np.expm1(test_predictions), 0, None)

print()
print("Prediction summary:")
print(f"  min price predicted  : ${final_prices.min():,.0f}")
print(f"  median price predicted: ${np.median(final_prices):,.0f}")
print(f"  max price predicted  : ${final_prices.max():,.0f}")
print(f"  any negative prices  : {(final_prices < 0).sum()}")

## 10.6 Sanity check: do predictions look realistic?

Before writing the submission file, we check that predictions are
in a reasonable range. The training data showed prices from
$7,500 to $142,000. Predictions wildly outside this range
would suggest something went wrong in the pipeline.

Also we check there are no NaN or negative values, the competition
scorer would reject those rows or give a bad score.

In [ ]:
print("=== Final prediction sanity check ===")
print()

# range check
print(f"Training price range : $7,500 to $142,000")
print(f"Predicted price range: ${final_prices.min():,.0f} to ${final_prices.max():,.0f}")
print()

# distribution check
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.hist(np.log1p(final_prices), bins=50, color="#4C72B0", edgecolor="white")
plt.xlabel("log(1 + predicted price)")
plt.title("Test prediction distribution (log scale)")

plt.subplot(1, 2, 2)
plt.hist(np.log1p(train_fe[TARGET]), bins=50, color="#55A868", edgecolor="white", alpha=0.7)
plt.xlabel("log(1 + actual price)")
plt.title("Training price distribution (log scale)")
plt.tight_layout()
plt.show()

print("If both histograms have similar shapes, predictions are sensible.")
print()

# NaN and negative check
nan_count = np.isnan(final_prices).sum()
neg_count = (final_prices < 0).sum()
print(f"NaN predictions      : {nan_count}  (should be 0)")
print(f"Negative predictions : {neg_count}  (should be 0)")

# 11. Generate Submission File

In [ ]:
# Build and verify submission file

sample_sub = pd.read_csv(
    "/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv"
)
print("Expected columns:", sample_sub.columns.tolist())
print("Expected rows   :", len(sample_sub))
print()

submission = pd.DataFrame({
    "TransactionID": test_fe["TransactionID"].values,
    "TargetValue"  : final_prices
})

# verify it matches the sample format
assert list(submission.columns) == list(sample_sub.columns), \
    "Column mismatch — check column names"
assert len(submission) == len(sample_sub), \
    f"Row mismatch — expected {len(sample_sub)}, got {len(submission)}"
assert submission["TargetValue"].isna().sum() == 0, \
    "NaN values found in predictions"

print("Submission verification passed")
print(f"Rows    : {len(submission)}")
print(f"Columns : {list(submission.columns)}")
print()
print("First 5 predictions:")
display(submission.head())

## 11.1 Write submission.csv

In [ ]:
submission.to_csv("submission.csv", index=False)
